In [ ]:
import os
import fitz  # PyMuPDF

# =========================================================================
# 🏆 【RAG 文字清洗優化模組】空間幾何腳註焊接器 (Spatial Footnote Linker)
# 功能說明:
#   針對 PDF 提取時頁尾腳註 (Footnotes) 與主體正文斷裂的 RAG 痛點，
#   透過空間坐標動態切分，強制將中斷的頁尾細節死數據焊接回當頁正文結尾。
# =========================================================================

def spatial_footnote_text_extractor(pdf_path):
    """
    輸入 PDF 絕對路徑，回傳內含正文與頁尾腳註完美焊接的標準 RAG Chunks 列表。
    全面提升 RAG 對「各國歷史之最」、「重大巨災損失細節」等高難度死數據的檢索命中率。
    """
    if not os.path.exists(pdf_path):
        print(f"⚠️ 提示：找不到目標檔案: {pdf_path}")
        return []
        
    doc = fitz.open(pdf_path)
    page_chunks = []
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        page_height = page.rect.height
        
        # 鎖定頁面底部 15% 的空間作為腳註防線
        footnote_boundary = page_height * 0.85  
        
        # 獲取頁面上所有文字塊的詳細坐標資訊
        blocks = page.get_text("blocks")
        main_body_text = ""
        footnote_text = ""
        
        for b in blocks:
            x0, y0, x1, y1, text_content, block_no, block_type = b
            clean_text = text_content.strip()
            if not clean_text:
                continue
                
            # 依據幾何空間坐標（Y軸）進行正文與腳註的分流
            if y0 > footnote_boundary:
                footnote_text += f" [Footnote Detail: {clean_text}]"
            else:
                main_body_text += f" {clean_text}"
        
        # 強制執行語義焊接：將腳註掛在正文屁股後面
        if footnote_text:
            final_content = f"{main_body_text}\n\n📢 [Linked Footnote For This Context]: {footnote_text}"
        else:
            final_content = main_body_text
            
        page_chunks.append({
            "page_content": f"[SOURCE: {os.path.basename(pdf_path)}_P{page_num+1}]\n{final_content}",
            "metadata": {
                "source_file": os.path.basename(pdf_path),
                "page": page_num + 1,
                "data_type": "text_with_welded_footnotes"
            }
        })
        
    return page_chunks

print("✅ 【空間幾何腳註焊接器】函數 `spatial_footnote_text_extractor` 已成功註冊！")
print("🚀 提示：文字組隊友可直接將此函數引入全組的 PDF 文本讀取 Pipeline 中。")